# gramlab · run 20 — a pretrained dense finder with the core-v1 key as verifier

Closes the question left by runs 16–19: does the structural work (ordered key, role verification) add anything **on top of a
real retriever**? Finder: `sentence-transformers/all-MiniLM-L6-v2` (22M parameters, CPU). Verifier: the core-v1 key index
(+bonus for sentences whose stored event matches the question's resolved event). Two tests, same as before:

1. the held-out AMALGUM paraphrase/reversal bench (run 17 construction): BM25 · dense · key · all combinations;
2. SQuAD v1.1 dev, sentence-level answer containment (run 16/18 construction).

CPU only, no GPU. Internet on (model download). Expected wall time ≈ 45–60 min, most of it MiniLM encoding 192k sentences.
If the `gramlab-core` notebook output is attached as input its `results/*.pkl` are reused; otherwise the linker and front
end are retrained here (+10 min).

In [ ]:
import os, sys, glob, subprocess, pathlib, json
WORK = pathlib.Path('/kaggle/working'); os.chdir(WORK)
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0
sh('git clone --depth 1 https://github.com/furevych/gramlab.git gramlab')
sys.path.insert(0, str(WORK / 'gramlab'))
GUM = WORK / 'data' / 'gum'; GUM.mkdir(parents=True, exist_ok=True)
for split in ('train', 'dev', 'test'):
    sh(f'curl -sSL -o {GUM}/en_gum-ud-{split}.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_English-GUM/master/en_gum-ud-{split}.conllu')
AMALGUM = 'data/amalgum/amalgum/*/dep/*.conllu'
if not glob.glob(AMALGUM):
    sh('GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/gucorpling/amalgum.git data/amalgum')
print('GUM', sorted(p.name for p in GUM.glob('*.conllu')), '| AMALGUM files', len(glob.glob(AMALGUM)))
# layer-0 preset: WordNet 3.0 files, read by gramlab.core.wordnet (no nltk needed)
if not pathlib.Path('data/wordnet/index.verb').exists():
    sh('curl -sSL -o data/wordnet.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip')
    import zipfile; zipfile.ZipFile('data/wordnet.zip').extractall('data')      # -> data/wordnet/
os.environ['WORDNET_DIR'] = str(WORK / 'data' / 'wordnet')
from gramlab.core.wordnet import WordNet; print('WordNet', WordNet().stats())
if not pathlib.Path('data/semcor/brown2').exists():
    sh('curl -sSL -o data/semcor.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/semcor.zip')
    zipfile.ZipFile('data/semcor.zip').extractall('data')                        # -> data/semcor/
SEMCOR = str(WORK / 'data' / 'semcor')
# affect lexicon: attach a Kaggle dataset containing NRC-VAD-Lexicon.txt (word, valence, arousal, dominance) to enable the affect row
VAD = next(iter(glob.glob('/kaggle/input/**/NRC-VAD-Lexicon*.txt', recursive=True) + glob.glob('/kaggle/input/**/*vad*.txt', recursive=True)), None)
print('VAD lexicon:', VAD)
# gazetteer from Wikipedia lead sentences: attach the gramlab-fullwiki notebook output as an input (work/bm25_full/passages.jsonl)
PASSAGES = next(iter(glob.glob('/kaggle/input/**/bm25_full/passages.jsonl', recursive=True) + glob.glob('/kaggle/input/**/passages.jsonl', recursive=True)), None)
GAZ = None
if PASSAGES:
    GAZ = str(WORK / 'data' / 'gazetteer.tsv')
    if not pathlib.Path(GAZ).exists():
        from gramlab.core.gazetteer import build; build(PASSAGES, GAZ, WordNet())
print('passages for gazetteer:', PASSAGES)
if PASSAGES:   # probe: what do the stored leads of a few problem titles look like?
    want = {'Russia', 'Duvall (surname)', 'Sam', 'Petitioner', 'Brahms', 'Court', 'Language'}
    with open(PASSAGES, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            if r.get('title') in want:
                print('LEAD', r['title'], '|', r.get('text', '')[:220].replace('\n', ' '))
                want.discard(r['title'])
                if not want: break

In [ ]:
# linker + live front end: reuse from attached gramlab-core output, else train
import shutil
RES = WORK / 'results'; RES.mkdir(exist_ok=True)
src = pathlib.Path('/kaggle/input/notebooks/tarasfur/gramlab-core/results')
if (src / 'linker.pkl').exists() and (src / 'frontend.pkl').exists():
    for f in ('linker.pkl', 'frontend.pkl'): shutil.copy(src / f, RES / f)
    print('reused linker/front end from gramlab-core output')
else:
    from gramlab.core import bench
    bench.run(str(GUM), AMALGUM, VAD, out_dir=str(RES), semcor=SEMCOR, gazetteer=GAZ)
sh('pip -q install -U sentence-transformers 2>/dev/null || true')
from sentence_transformers import SentenceTransformer
DENSE = 'sentence-transformers/all-MiniLM-L6-v2'
_ = SentenceTransformer(DENSE, device='cpu'); print('encoder ready')

In [ ]:
# 1) held-out paraphrase / reversal bench: BM25 · dense · key · soft map (untrained) · combinations  (~35 min)
from gramlab.core import softmap
board = softmap.run(AMALGUM, SEMCOR, str(RES / 'frontend.pkl'), str(RES / 'linker.pkl'), out_dir=str(RES), n_queries=2000,
                    variants=('roles/synsets/untrained',), key_bonus=1.0, dense=DENSE)
import json; print(json.dumps(board['systems'], indent=1))

In [ ]:
# 2) SQuAD dev: BM25 · dense · key verification · combinations  (~15 min)
sh('git clone --depth 1 https://github.com/rajpurkar/SQuAD-explorer.git data/squad')
from gramlab.core import softmap_external
ext = softmap_external.run('data/squad/dataset/dev-v1.1.json', str(RES / 'frontend.pkl'), str(RES / 'linker.pkl'),
                           str(RES / 'softmap.npz'), SEMCOR, out_dir=str(RES), dense=DENSE, key_bonus=1.0)

In [ ]:
# sensitivity: a weaker key bonus (run 16 found the key's candidates wrong more often than BM25's on SQuAD)
ext_half = softmap_external.run('data/squad/dataset/dev-v1.1.json', str(RES / 'frontend.pkl'), str(RES / 'linker.pkl'),
                                str(RES / 'softmap.npz'), SEMCOR, out_dir=str(RES / 'bonus05'), dense=DENSE, key_bonus=0.5)

## Reading it
- `dense` vs `bm25`: how much a real finder gains over words on each test.
- `hybrid dense+key` / `three-way` vs `dense`: the verifier's contribution — the number this project's result rests on.
- `source_above_decoy`: only systems with the key (or role slots) separate *X called Y* from *Y called X*; a dense encoder alone mostly cannot.
Record the numbers in `RESULTS.md` as run 20.